In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact
# from IPython.display import clear_output # Importar clear_output - removido para mostrar ambos gráficos

# Concentradores de esfuerzo en fatiga:
#   False -> Kf y Kfs se calculan: Kf = 1 + q(Kt-1), con q de Neuber (constante de Neuber 'sqrt(a)' en funcion de Sut)
#   True  -> se usan los valores redondeados de la hoja manual: Kf = 1.539, Kfs = 1.369 (q = 0.77 y 0.82)
# Kt y Kts son lecturas de las graficas de Peterson/Shigley (G-2 y G-3) para D/d y r/d dados: son entradas (sliders).
USAR_KF_HOJA = False

# --- Función Principal: calculate_and_plot_fatigue_interactive ---
# Esta función realiza todos los cálculos de fatiga, desde la entrada de datos
# hasta la determinación de los factores de seguridad, y luego grafica el diagrama de Smith.

def calculate_and_plot_fatigue_interactive(
    D, d, r, Sut, Sy, acabado_opc, confiabilidad, Tmax, Tmin, Mmax, Mmin, Kt=1.70, Kts=1.45
):
    # clear_output(wait=True) # Limpiar la salida anterior antes de generar una nueva - removido para mostrar ambos gráficos
    print("--- Calculadora de Fatiga en Ejes Escalonados ---")

    # 1. Datos de Entrada (obtenidos de los sliders)
    # D, d, r, Sut, Sy, acabado_opc, confiabilidad, Tmax, Tmin, Mmax, Mmin
    # son ahora argumentos de la función.

    # 2. Análisis de Cargas (Alternantes y Medias)
    Ta = abs((Tmax - Tmin) / 2)
    Tm = (Tmax + Tmin) / 2
    Ma = abs((Mmax - Mmin) / 2)
    Mm = (Mmax + Mmin) / 2

    # 3. Factores de Marin
    ka_params = {1: (1.58, -0.085), 2: (4.51, -0.265), 3: (57.7, -0.718), 4: (272, -0.995)}
    a, b = ka_params[acabado_opc]
    ka = a * (Sut**b)

    # kb - Factor de tamaño (válido para 2.79 <= d <= 51 mm)
    # Se asume que el usuario mantendrá 'd' en un rango razonable con el slider.
    kb = 1.24 * (d**-0.107)

    # ke - Factor de confiabilidad
    ke_dict = {0.5: 1.0, 0.9: 0.897, 0.95: 0.868, 0.99: 0.814, 0.999: 0.753}
    ke = ke_dict.get(confiabilidad, 0.814) # Usa 0.814 si la confiabilidad no está en el diccionario

    # Límite de Resistencia a la Fatiga Modificado (Se)
    Se_prime = 0.5 * Sut
    Se = ka * kb * ke * Se_prime

    # 4. Concentración de Esfuerzos
    # Kt y Kts: factores teóricos leídos de las gráficas (entradas). Sensibilidad a la entalla de Neuber:
    #   flexión/axial (340-1700 MPa): sqrt(a) = 1.24 - 2.25e-3 Sut + 1.60e-6 Sut^2 - 4.11e-10 Sut^3   [sqrt(mm)]
    #   torsión       (340-1500 MPa): sqrt(a) = 0.958 - 1.83e-3 Sut + 1.43e-6 Sut^2 - 4.11e-10 Sut^3  [sqrt(mm)]
    sqrt_a_b = 1.24 - 2.25e-3*Sut + 1.60e-6*Sut**2 - 4.11e-10*Sut**3
    sqrt_a_t = 0.958 - 1.83e-3*Sut + 1.43e-6*Sut**2 - 4.11e-10*Sut**3
    if r <= 0.01: # Evitar división por cero o valores muy pequeños de r
        q, qs = 1.0, 1.0 # Máxima sensibilidad
    else:
        q = 1 / (1 + sqrt_a_b/np.sqrt(r))
        qs = 1 / (1 + sqrt_a_t/np.sqrt(r))
    Kf = 1 + q * (Kt - 1)
    Kfs = 1 + qs * (Kts - 1)
    if USAR_KF_HOJA:
        Kf, Kfs = 1.539, 1.369

    # 5. Esfuerzos Nominales (MPa)
    if d <= 0.01: # Evitar división por cero o valores muy pequeños de d
        print("Error: El diámetro d debe ser mayor que cero.")
        return
    def stress_flex(M): return (32 * M) / (np.pi * (d/1000)**3) / 1e6
    def stress_tor(T): return (16 * T) / (np.pi * (d/1000)**3) / 1e6

    sig_a = stress_flex(Ma)
    sig_m = stress_flex(Mm)
    tau_a = stress_tor(Ta)
    tau_m = stress_tor(Tm)

    # 6. Esfuerzos Equivalentes de Von Mises (Fatiga)
    sig_prime_a = np.sqrt((Kf * sig_a)**2 + 3 * (Kfs * tau_a)**2)
    sig_prime_m = np.sqrt((Kf * sig_m)**2 + 3 * (Kfs * tau_m)**2)

    # 7. Factores de Seguridad
    ny_langer = Sy / (sig_prime_a + sig_prime_m) if (sig_prime_a + sig_prime_m) != 0 else np.inf
    n_goodman = 1 / (sig_prime_a/Se + sig_prime_m/Sut) if (sig_prime_a/Se + sig_prime_m/Sut) != 0 else np.inf
    n_soderberg = 1 / (sig_prime_a/Se + sig_prime_m/Sy) if (sig_prime_a/Se + sig_prime_m/Sy) != 0 else np.inf

    asme_denom_sq = (sig_prime_a/Se)**2 + (sig_prime_m/Sy)**2
    n_asme = np.sqrt(1 / asme_denom_sq) if asme_denom_sq > 0 else np.inf

    if sig_prime_m == 0 or sig_prime_a == 0 or Sut == 0 or Se == 0: # Evitar división por cero
        n_gerber = np.inf
    else:
        term_b = (2 * sig_prime_m * Se) / (sig_prime_a * Sut)
        sqrt_arg = 1 + term_b**2
        if sqrt_arg < 0:
            n_gerber = np.inf
        else:
            n_gerber = (1/2) * (Sut/sig_prime_m)**2 * (sig_prime_a/Se) * (-1 + np.sqrt(sqrt_arg))

    # Resultados
    print("\n" + "="*30)
    print("RESULTADOS DEL ANÁLISIS")
    print("="*30)
    print(f"Factor Marin Superficie (ka): {ka:.3f}")
    print(f"Factor Marin Tamaño (kb):     {kb:.3f}")
    print(f"Factor Marin Confiabilidad (ke): {ke:.3f}")
    print(f"Kf = {Kf:.3f}, Kfs = {Kfs:.3f}")
    print(f"Límite de Fatiga (Se): {Se:.2f} MPa")
    print(f"Esfuerzo Von Mises Alternante: {sig_prime_a:.2f} MPa")
    print(f"Esfuerzo Von Mises Medio: {sig_prime_m:.2f} MPa")
    # Esfuerzo estático máximo en la muesca con Kt y Kts (comparación con el modelo de elementos finitos)
    s_max = stress_flex(Mmax); t_max = stress_tor(Tmax)
    print(f"Von Mises estático en la muesca (Kt, Kts): {np.sqrt((Kt*s_max)**2 + 3*(Kts*t_max)**2):.2f} MPa")
    print("-" * 30)
    print(f"n Langer (Fluencia):  {ny_langer:.3f}")
    print(f"n Goodman:            {n_goodman:.3f}")
    print(f"n Soderberg:          {n_soderberg:.3f}")
    print(f"n ASME Elíptica:      {n_asme:.3f}")
    print(f"n Gerber:             {n_gerber:.3f}")

    # Llamar a las funciones de graficación
    # Crear una única figura con dos subplots
    fig, axes = plt.subplots(2, 1, figsize=(10, 10)) # Dos filas, una columna

    plot_smith_diagram(Se, Sut, Sy, sig_prime_a, sig_prime_m, ax=axes[0])
    plot_sn_diagram(Se, Sut, Sy, sig_prime_a, sig_prime_m, ax=axes[1])

    plt.tight_layout() # Ajustar el diseño para evitar superposiciones
    plt.show()


# --- Función de Graficación: plot_smith_diagram ---

def plot_smith_diagram(Se, Sut, Sy, sig_prime_a, sig_prime_m, ax):
    # plt.close() # Cierra la figura anterior para permitir la actualización en tiempo real - movido al subplot
    # plt.figure(figsize=(10, 8)) # Creado en el subplot principal
    ax.set_title('Diagrama de Smith (Sm vs Sa) con Criterios de Fatiga')
    ax.set_xlabel('Esfuerzo Medio Equivalente (Sm) [MPa]')
    ax.set_ylabel('Esfuerzo Alternante Equivalente (Sa) [MPa]')
    ax.grid(True, linestyle='--', alpha=0.7)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)

    # Generar un rango de esfuerzos medios para las líneas de falla
    max_x_val = max(Sut, Sy, sig_prime_m * 1.5, 10) if Sut > 0 and Sy > 0 else 1000
    Sm_vals = np.linspace(0, max_x_val, 200)

    # Línea de Falla de Goodman
    Sa_goodman = Se * (1 - Sm_vals / Sut) if Sut != 0 else np.zeros_like(Sm_vals)
    ax.plot(Sm_vals, Sa_goodman, label='Goodman', color='blue', linestyle='-')

    # Línea de Falla de Soderberg
    Sa_soderberg = Se * (1 - Sm_vals / Sy) if Sy != 0 else np.zeros_like(Sm_vals)
    ax.plot(Sm_vals, Sa_soderberg, label='Soderberg', color='green', linestyle='--')

    # Parábola de Falla de Gerber
    Sa_gerber = Se * (1 - (Sm_vals / Sut)**2) if Sut != 0 else np.zeros_like(Sm_vals)
    ax.plot(Sm_vals, Sa_gerber, label='Gerber', color='red', linestyle='-.')

    # Elipse de Falla ASME
    Sa_asme = np.zeros_like(Sm_vals)
    if Sy != 0: # Evitar división por cero
        for i, Sm_val in enumerate(Sm_vals):
            arg = 1 - (Sm_val / Sy)**2
            if arg >= 0:
                Sa_asme[i] = Se * np.sqrt(arg)
            else:
                Sa_asme[i] = 0 # O un valor que no se grafique
    ax.plot(Sm_vals, Sa_asme, label='ASME Elíptica', color='purple', linestyle=':')

    # Línea de Falla de Langer (Fluencia)
    Sa_langer = Sy - Sm_vals
    # Solo graficar la parte donde Sa >= 0
    Sa_langer[Sa_langer < 0] = np.nan # Usar NaN para no graficar la parte negativa
    ax.plot(Sm_vals, Sa_langer, label='Langer (Fluencia)', color='brown', linestyle=':')

    # Trazar el punto de operación
    ax.plot(sig_prime_m, sig_prime_a, 'o', color='orange', markersize=8, label='Punto de Operación', zorder=5)

    # Dibujar una flecha desde el origen hasta el punto de operación
    ax.annotate('', xy=(sig_prime_m, sig_prime_a), xytext=(0, 0),
                 arrowprops=dict(facecolor='black', shrink=0.05, width=0.5, headwidth=5))

    # Ajustar límites de los ejes para una mejor visualización
    max_y_val = max(Se, sig_prime_a * 1.2) * 1.1 if Se > 0 or sig_prime_a > 0 else 200
    ax.set_xlim(0, max_x_val * 1.1)
    ax.set_ylim(0, max_y_val)
    ax.legend(loc='upper right')
    # plt.tight_layout() # Movido al subplot principal
    # plt.show() # Movido al subplot principal

# --- Función de Graficación: plot_sn_diagram ---

def plot_sn_diagram(Se, Sut, Sy, sig_prime_a, sig_prime_m, ax):
    # plt.close() # Cierra cualquier figura S-N anterior - movido al subplot
    # plt.figure(figsize=(10, 8)) # Creado en el subplot principal
    ax.set_title('Diagrama S-N (Amplitud de Esfuerzo vs. Ciclos a la Falla)')
    ax.set_xlabel('Número de Ciclos a la Falla (N)')
    ax.set_ylabel('Amplitud de Esfuerzo Equivalente (Sa_eq) [MPa]')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.grid(True, which="both", ls="--", alpha=0.7)

    # --- Definir la Curva S-N (Ecuación de Basquin aproximada para acero) ---
    # Para aceros, se asume que la resistencia a la fatiga a 10^3 ciclos es 0.9 * Sut
    # y la resistencia a la fatiga a 10^6 ciclos es Se.
    N_low = 1000  # Ciclos para la resistencia a baja fatiga
    S_low = 0.9 * Sut  # Resistencia a la fatiga a N_low

    N_high = 10**6  # Ciclos para la resistencia a alta fatiga (límite de fatiga)
    S_high = Se  # Límite de fatiga modificado

    # Calcular los parámetros A y b de la ecuación de Basquin: Sa = A * N^b
    if S_low <= 0 or S_high <= 0 or N_low >= N_high or Sut == 0: # Añadir Sut=0 check
        print("Advertencia: No se puede generar la curva S-N con valores de resistencia cero o negativos, o Sut inválido.")
        # Plot a basic line to prevent errors if S-N curve cannot be defined
        ax.plot([1, 10**7], [Se, Se], 'k--', label='Límite de Fatiga (Se)')
        ax.legend()
        # plt.show() # Movido al subplot principal
        return

    # Calcular b y logA solo si los denominadores son válidos
    if (np.log10(N_high) - np.log10(N_low)) != 0: # Evitar división por cero
        b = (np.log10(S_high) - np.log10(S_low)) / (np.log10(N_high) - np.log10(N_low))
        logA = np.log10(S_low) - b * np.log10(N_low)
        A = 10**logA
    else:
        b = 0 # Caso de línea horizontal
        A = S_high # Si N_low == N_high, o si log(N) es cero

    # Generar puntos para la curva S-N
    N_cycles = np.logspace(1, 7, 500) # Desde 10^1 hasta 10^7 ciclos

    # Asegurarse de que A no sea cero y b no cause un error
    if A == 0: # Si A es cero, la curva de fatiga es cero.
        Sn_curve_Sa = np.zeros_like(N_cycles)
    else:
        # Para N < N_high, usar Basquin. Para N >= N_high, usar Se.
        Sn_curve_Sa_calculated = A * (N_cycles**b)
        Sn_curve_Sa = np.where(N_cycles < N_high, Sn_curve_Sa_calculated, Se)

    # Asegurarse de que la curva no suba por encima de Sut (realista)
    Sn_curve_Sa = np.minimum(Sn_curve_Sa, Sut)

    ax.plot(N_cycles, Sn_curve_Sa, 'k-', label='Curva S-N (Cero Esfuerzo Medio)')

    # --- Calcular Amplitudes de Esfuerzo Equivalentes (Sa_eq) y proyectarlas en la curva S-N ---
    # Se utilizan los criterios de Goodman, Soderberg, Gerber y ASME para convertir
    # el punto de operación (sig_prime_a, sig_prime_m) a una amplitud de esfuerzo equivalente
    # totalmente invertida (Sa_eq) que se proyecta en la curva S-N.

    # Función auxiliar para calcular N a partir de Sa_eq (invirtiendo la ecuación de Basquin)
    def calculate_N_from_Sa(Sa_eq_val, A_val, b_val, Se_val, N_high_val, Sut_val):
        if Sa_eq_val <= Se_val:
            return 10**7 # Vida infinita para Sa_eq <= Se
        if Sa_eq_val > Sut_val:
            return 1 # Falla casi inmediata si Sa_eq excede Sut
        if A_val == 0 or b_val == 0: # Evitar división por cero si la curva es degenerada
            return 10**7
        try:
            return (Sa_eq_val / A_val)**(1/b_val)
        except ZeroDivisionError:
            return 10**7 # En caso de que b sea 0
        except OverflowError:
            return 10**7 # Si el exponente 1/b es muy grande

    # Goodman
    if Sut != 0 and (1 - sig_prime_m / Sut) > 0: # Evitar división por cero o negativo
        Sa_eq_goodman = sig_prime_a / (1 - sig_prime_m / Sut)
        N_goodman = calculate_N_from_Sa(Sa_eq_goodman, A, b, Se, N_high, Sut)
        ax.plot(N_goodman, Sa_eq_goodman, 'o', color='blue', markersize=8, label=f'Goodman (N={N_goodman:.0f} ciclos)')
    else:
        print("Advertencia: Criterio de Goodman no aplicable para el diagrama S-N (Sut=0 o sig_prime_m/Sut >= 1).")

    # Soderberg
    if Sy != 0 and (1 - sig_prime_m / Sy) > 0: # Evitar división por cero o negativo
        Sa_eq_soderberg = sig_prime_a / (1 - sig_prime_m / Sy)
        N_soderberg = calculate_N_from_Sa(Sa_eq_soderberg, A, b, Se, N_high, Sut)
        ax.plot(N_soderberg, Sa_eq_soderberg, 'o', color='green', markersize=8, label=f'Soderberg (N={N_soderberg:.0f} ciclos)')
    else:
        print("Advertencia: Criterio de Soderberg no aplicable para el diagrama S-N (Sy=0 o sig_prime_m/Sy >= 1).")

    # Gerber
    if Sut != 0 and (1 - (sig_prime_m / Sut)**2) > 0: # Evitar división por cero o negativo
        Sa_eq_gerber = sig_prime_a / (1 - (sig_prime_m / Sut)**2)
        N_gerber = calculate_N_from_Sa(Sa_eq_gerber, A, b, Se, N_high, Sut)
        ax.plot(N_gerber, Sa_eq_gerber, 'o', color='red', markersize=8, label=f'Gerber (N={N_gerber:.0f} ciclos)')
    else:
        print("Advertencia: Criterio de Gerber no aplicable para el diagrama S-N (Sut=0 o (sig_prime_m/Sut)^2 >= 1).")

    # ASME Elíptica
    if Sy != 0 and (1 - (sig_prime_m / Sy)**2) > 0: # Evitar división por cero o negativo
        Sa_eq_asme = sig_prime_a / np.sqrt(1 - (sig_prime_m / Sy)**2)
        N_asme = calculate_N_from_Sa(Sa_eq_asme, A, b, Se, N_high, Sut)
        ax.plot(N_asme, Sa_eq_asme, 'o', color='purple', markersize=8, label=f'ASME Elíptica (N={N_asme:.0f} ciclos)')
    else:
        print("Advertencia: Criterio ASME Elíptica no aplicable para el diagrama S-N (Sy=0 o (sig_prime_m/Sy)^2 >= 1).")

    ax.legend(loc='lower left')
    ax.set_xlim(10, 10**7)
    max_sa_plot = max(Sut * 1.1, sig_prime_a * 1.5, Se * 1.5, 10) # Asegurar un valor mínimo para el eje y
    ax.set_ylim(Se * 0.8, max_sa_plot) # Ajustar el eje y para mostrar el rango de Sa relevante
    # plt.tight_layout() # Movido al subplot principal
    # plt.show() # Movido al subplot principal


# --- Configuración y Ejecución del Análisis Interactivo ---
# Se define un widget interactivo para la función calculate_and_plot_fatigue_interactive

interact(
    calculate_and_plot_fatigue_interactive,
    D=widgets.FloatSlider(min=20, max=100, step=1, value=50, description='D (mm)'),
    d=widgets.FloatSlider(min=10, max=90, step=1, value=30, description='d (mm)'),
    r=widgets.FloatSlider(min=0.1, max=10, step=0.1, value=3, description='r (mm)'),
    Sut=widgets.FloatSlider(min=300, max=800, step=10, value=460, description='Sut (MPa)'),
    Sy=widgets.FloatSlider(min=150, max=600, step=10, value=250, description='Sy (MPa)'),
    acabado_opc=widgets.Dropdown(options=[(f"{i}. {['Rectificado', 'Maquinado/Laminado Frío', 'Laminado Caliente', 'Asforjado'][i-1]}", i) for i in range(1, 5)], value=2, description='Acabado'),
    confiabilidad=widgets.Dropdown(options=[0.5, 0.9, 0.95, 0.99, 0.999], value=0.99, description='Confiabilidad'),
    Tmax=widgets.FloatSlider(min=-200, max=200, step=5, value=130, description='Tmax (N-m)'),
    Tmin=widgets.FloatSlider(min=-200, max=200, step=5, value=-80, description='Tmin (N-m)'),
    Mmax=widgets.FloatSlider(min=0, max=200, step=5, value=130, description='Mmax (N-m)'),
    Mmin=widgets.FloatSlider(min=0, max=200, step=5, value=13, description='Mmin (N-m)'),
    Kt=widgets.FloatSlider(min=1.0, max=3.0, step=0.01, value=1.70, description='Kt (flexión)'),
    Kts=widgets.FloatSlider(min=1.0, max=3.0, step=0.01, value=1.45, description='Kts (torsión)')
);